# Olist Delivery Delay Model Report

This notebook contains EDA, model training results, and a short demonstration of the declarative pipeline that produced the Gold feature table and trained a GBT model.

Sections:
1. Run pipeline (or confirm it was run)
2. Inspect Gold feature table
3. Exploratory Data Analysis (EDA)
4. Model diagnostics and predictions
5. Feature importance and findings

In [ ]:
# NOTE: Execute the pipeline beforehand if not already run:
# !python ../scripts/train_olist.py --config ../config/olist_ml_train.yaml

# Setup: Spark + Delta + MLflow
from engine.config_parser import load_pipeline_config
from engine.runner import get_spark_session
import mlflow
import mlflow.tracking
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

cfg = load_pipeline_config('config/olist_ml_train.yaml')
spark = get_spark_session('OlistNotebook')

gold_path = cfg.gold.target_delta_path
print('Loading Gold features from:', gold_path)
df_gold = spark.read.format('delta').load(gold_path)
# Convert a sample to pandas for quick EDA (limit to 10000 rows)
df_pd = df_gold.limit(10000).toPandas()
df_pd.head()

## 1 — Quick Dataset Overview
Show schema, basic counts, and target distribution.

In [ ]:
# Schema and counts
print('Spark schema:')
df_gold.printSchema()
print('Total Gold records:', df_gold.count())

# Pandas summary statistics for numeric features and target
numeric_cols = [c for c in df_pd.select_dtypes(include=['number']).columns.tolist()]
print('Numeric columns:', numeric_cols)
display(df_pd[numeric_cols].describe().round(3))

# Target distribution
target = cfg.ml_stage.target_column if cfg.ml_stage else 'actual_delay_days'
plt.figure(figsize=(8,4))
sns.histplot(df_pd[target].dropna(), bins=50, kde=True)
plt.title('Distribution of target: ' + target)
plt.xlabel(target)
plt.show()

## 2 — Correlations & Relationships
Look for simple linear relationships between numeric features and the target.

In [ ]:
# Pairwise scatterplots for numeric features vs target (sampled)
sample = df_pd.sample(n=min(5000, len(df_pd)), random_state=42)
features_to_plot = [f for f in cfg.ml_stage.numeric_features] if cfg.ml_stage else []
for f in features_to_plot:
    plt.figure(figsize=(6,3))
    sns.scatterplot(x=sample[f], y=sample[target], alpha=0.4, s=10)
    plt.title(f + ' vs ' + target)
    plt.xlabel(f)
    plt.ylabel(target)
    plt.show()

## 3 — Model Diagnostics
Load the latest MLflow run, print metrics, and generate prediction diagnostics (predicted vs actual, residuals).

In [ ]:
# Load latest MLflow run and metrics
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name(cfg.mlflow.experiment_name)
if exp is None:
    raise RuntimeError(f'MLflow experiment not found: {cfg.mlflow.experiment_name}')
runs = client.search_runs(exp.experiment_id, order_by=["attributes.start_time DESC"], max_results=1)
if not runs:
    raise RuntimeError('No runs found in experiment')
run = runs[0]
run_id = run.info.run_id
print('Using MLflow run:', run_id)
print('Logged metrics:', run.data.metrics)

# Try to load the saved Spark PipelineModel artifact and run predictions on a sample
pred_sample = None
try:
    model_uri = f'runs:/{run_id}/gbt_delivery_model'
    print('Loading model from', model_uri)
    pipeline_model = mlflow.spark.load_model(model_uri)
    # Transform a small Spark sample for diagnostics
    df_sample = df_gold.limit(2000)
    preds = pipeline_model.transform(df_sample)
    preds_pd = preds.select(cfg.ml_stage.target_column, 'predicted_delay_days').toPandas()
    pred_sample = preds_pd.dropna()
    print('Predictions sample size:', len(pred_sample))
except Exception as e:
    print('Could not load or score model artifact:', e)

if pred_sample is not None and not pred_sample.empty:
    # Predicted vs Actual scatter
    plt.figure(figsize=(6,6))
    sns.scatterplot(x=pred_sample[cfg.ml_stage.target_column], y=pred_sample['predicted_delay_days'], alpha=0.4, s=10)
    plt.plot(plt.gca().get_xlim(), plt.gca().get_xlim(), color='red', linestyle='--')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Predicted vs Actual')
    plt.show()

    # Residuals histogram
    residuals = pred_sample['predicted_delay_days'] - pred_sample[cfg.ml_stage.target_column]
    plt.figure(figsize=(6,3))
    sns.histplot(residuals, bins=50, kde=True)
    plt.title('Residuals (predicted - actual)')
    plt.show()

## 4 — Feature Importance
Extract feature importances from the trained GBT model (from the PipelineModel) and plot them.

In [ ]:
# Attempt to extract feature importances from the loaded pipeline_model
try:
    # Find VectorAssembler inputs and GBT stage
    assembler_inputs = None
    gbt_stage = None
    for stage in pipeline_model.stages:
        cls_name = stage.__class__.__name__
        if hasattr(stage, 'getInputCols') and assembler_inputs is None:
            assembler_inputs = list(stage.getInputCols())
        if hasattr(stage, 'featureImportances'):
            gbt_stage = stage
    if gbt_stage is not None and assembler_inputs is not None:
        importances = list(gbt_stage.featureImportances)
        # Map and sort
        feat_imp = list(zip(assembler_inputs, importances))
        feat_imp_sorted = sorted(feat_imp, key=lambda x: x[1], reverse=True)
        names = [n for n, s in feat_imp_sorted]
        scores = [s for n, s in feat_imp_sorted]

        plt.figure(figsize=(8, max(4, len(names) * 0.4)))
        sns.barplot(x=scores, y=names, palette='viridis')
        plt.title('Feature Importances (GBT)')
        plt.xlabel('Importance')
        plt.tight_layout()
        plt.show()
    else:
        print('Could not locate GBT stage or assembler inputs in pipeline_model.')
except Exception as e:
    print('Error extracting feature importances:', e)

In [ ]:
# Load and display the feature importance chart from MLflow artifacts
from IPython.display import Image, display
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name(cfg.mlflow.experiment_name)
if exp is None:
    print('MLflow experiment not found:', cfg.mlflow.experiment_name)
else:
    runs = client.search_runs(exp.experiment_id, order_by=["attributes.start_time DESC"], max_results=1)
    if not runs:
        print('No runs found in experiment')
    else:
        run = runs[0]
        run_id = run.info.run_id
        print('Displaying feature importance from run:', run_id)
        # look for image under 'charts' artifact folder
        try:
            arts = client.list_artifacts(run_id, 'charts')
            pngs = [a.path for a in arts if a.path.lower().endswith('.png')]
            if pngs:
                local = client.download_artifacts(run_id, pngs[0])
                display(Image(filename=local))
            else:
                print('No PNG chart found under artifacts/charts for this run.')
        except Exception as e:
            print('Error while fetching artifact:', e)

### Interpretation of Feature Importances
- The bar chart above shows the relative importance assigned by the trained GBT model to each input feature.
- Top features indicate which predictors the model relied on most; treat these as ‘relative importance’ signals rather than causal proof.
- Given the low overall R² reported in the diagnostics, the model explains only a small fraction of variance in delivery delay — so even the top features may not produce strong predictive performance on their own.
- Suggested next steps: expand feature set (distance, courier, time-of-day), add transformations and outlier handling, and re-evaluate feature importances after retraining.